# Colab 37 - symbol statistics of the three representations

**Why:** supervisor notes 33/34 reject the adjectives *diffuse* / *concentrated* in Section 3.4 and
ask for a number. This produces that number, and the appendix figures that go with it (note 32).

**What it does NOT do:** no model, no training, no evaluation. Descriptive statistics only.

**Design point:** the collections are rebuilt with the *same* filter as the run of record - same
`[50, 200]`, same two exempted domains - so the statistics describe the collections that are actually
evaluated, not the raw files. Three assertions stop the notebook if the sizes drift from
10,501 / 10,497 / 10,501.

## 1. Setup - clone the repository and locate the data

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')


In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')


In [ ]:
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt

# --- IDENTICAL to the run of record. Do not change these without changing the thesis. ---
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
MIN_LEN, MAX_LEN = 50, 200
RESCUED = {'4z0mC02', '3qkaE02'}   # kept, per the 2026-08-24 decision; disclosed in Section 3.4

AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s)
is_ss = lambda s: all(c in SS_SET for c in s)

# locked palette (memory/deck_color_scheme): 3Di blue / SS red / AA grey
COLOUR = {'AA': '#7f7f7f', 'SS': '#d62728', '3Di': '#1f77b4'}


## 2. Rebuild the three collections (identical filter to the run of record)

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')

def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq)
            and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))

id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq'])              if _valid(s, is_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq'])              if _valid(s, is_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)}

COLLECTION = {'AA': list(id_to_aa.values()),
              'SS': list(id_to_ss.values()),
              '3Di': list(id_to_3di.values())}
ALPHABET   = {'AA': AA_ALPHABET, 'SS': SS_ALPHABET, '3Di': AA_ALPHABET}

print('Collection sizes  (run of record: AA 10,501 / SS 10,497 / 3Di 10,501)')
for r in ['AA', 'SS', '3Di']:
    n = len(COLLECTION[r]); tot = sum(len(s) for s in COLLECTION[r])
    print(f'  {r:<4} sequences = {n:>6,}   symbols = {tot:>9,}')
assert len(COLLECTION['AA']) == 10_501, 'AA collection differs from the run of record - STOP and check'
assert len(COLLECTION['SS']) == 10_497, 'SS collection differs from the run of record - STOP and check'
assert len(COLLECTION['3Di']) == 10_501, '3Di collection differs from the run of record - STOP and check'
print('\nMatches the run of record.')


## 3. Letter-frequency profiles and entropy

`entropy_bits` is Shannon entropy over the symbol distribution. `normalised_entropy` divides by
$\log_2 k$, so 1.0 means every symbol is equally likely and lower means the mass concentrates on
fewer symbols - this is the number that replaces "diffuse" and "concentrated".

Note that `top5_share` is only meaningful for the two 20-symbol alphabets; SS has three symbols, so
its top-5 share is 1.0 by construction. Quote `normalised_entropy` when comparing all three.

In [ ]:
from collections import Counter

def profile(seqs, alphabet):
    obs = Counter()
    for s in seqs:
        obs.update(s)
    counts = pd.Series({c: obs.get(c, 0) for c in alphabet}, dtype=np.int64)
    unknown = {c: n for c, n in obs.items() if c not in set(alphabet)}
    if unknown:
        print(f'  ! symbols outside the declared alphabet, not counted: {unknown}')
    p = counts / counts.sum()
    return counts, p

def entropy_bits(p):
    q = p[p > 0]
    return float(-(q * np.log2(q)).sum())

rows, PROF = [], {}
for r in ['AA', 'SS', '3Di']:
    counts, p = profile(COLLECTION[r], ALPHABET[r])
    PROF[r] = p
    H = entropy_bits(p)
    k = len(ALPHABET[r])
    top5 = float(p.sort_values(ascending=False).head(5).sum())
    rows.append(dict(representation=r,
                     alphabet_size=k,
                     sequences=len(COLLECTION[r]),
                     symbols=int(counts.sum()),
                     entropy_bits=round(H, 3),
                     max_entropy_bits=round(np.log2(k), 3),
                     normalised_entropy=round(H / np.log2(k), 3),
                     effective_alphabet=round(2 ** H, 2),
                     top5_share=round(top5, 3),
                     most_frequent=p.idxmax(),
                     most_frequent_share=round(float(p.max()), 3)))

SYMSTATS = pd.DataFrame(rows)
SYMSTATS


## 4. First-order transition probabilities

The conditional entropy $H(X_{t+1} \mid X_t)$ measures how much uncertainty about the next symbol
remains once the current one is known. The difference from the first-order entropy is the mutual
information: how much of the sequence structure lives in *order* rather than in composition.

In [ ]:
def transition(seqs, alphabet):
    idx = {c: i for i, c in enumerate(alphabet)}
    k = len(alphabet)
    M = np.zeros((k, k), dtype=np.int64)
    for s in seqs:
        a = [idx[c] for c in s if c in idx]
        if len(a) > 1:
            np.add.at(M, (np.array(a[:-1]), np.array(a[1:])), 1)
    P = M / np.maximum(M.sum(1, keepdims=True), 1)
    return M, pd.DataFrame(P, index=list(alphabet), columns=list(alphabet))

TRANS = {}
cond_rows = []
for r in ['AA', 'SS', '3Di']:
    M, P = transition(COLLECTION[r], ALPHABET[r])
    TRANS[r] = P
    # conditional entropy H(X_t+1 | X_t), weighted by the observed state distribution
    w = M.sum(1) / M.sum()
    Hc = float(sum(w[i] * entropy_bits(P.iloc[i]) for i in range(len(w))))
    H1 = float(SYMSTATS.loc[SYMSTATS.representation == r, 'entropy_bits'].iloc[0])
    cond_rows.append(dict(representation=r,
                          entropy_bits=round(H1, 3),
                          conditional_entropy_bits=round(Hc, 3),
                          mutual_information_bits=round(H1 - Hc, 3)))

COND = pd.DataFrame(cond_rows)
print('Order matters more where the mutual information is larger:')
COND


## 5. Figures (appendix, note 32) - locked palette

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
for ax, r in zip(axes, ['AA', 'SS', '3Di']):
    p = PROF[r].sort_values(ascending=False)
    ax.bar(range(len(p)), p.values, color=COLOUR[r])
    ax.set_xticks(range(len(p))); ax.set_xticklabels(p.index, fontsize=8)
    ax.set_title(f'{r}  (H = {entropy_bits(PROF[r]):.2f} bits)')
    ax.set_ylabel('relative frequency' if r == 'AA' else '')
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('colab37_letter_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, r in zip(axes, ['AA', 'SS', '3Di']):
    im = ax.imshow(TRANS[r].values, cmap='magma', vmin=0,
                   vmax=float(np.percentile(TRANS[r].values, 99)))
    ax.set_title(f'{r} transition probabilities')
    ax.set_xticks(range(len(TRANS[r]))); ax.set_xticklabels(TRANS[r].columns, fontsize=7)
    ax.set_yticks(range(len(TRANS[r]))); ax.set_yticklabels(TRANS[r].index, fontsize=7)
    ax.set_xlabel('next symbol'); ax.set_ylabel('current symbol' if r == 'AA' else '')
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('colab37_transitions.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Save everything and print the Section 3.4 sentence

In [ ]:
SYMSTATS.to_csv('colab37_symbol_statistics.csv', index=False)
COND.to_csv('colab37_conditional_entropy.csv', index=False)
for r in ['AA', 'SS', '3Di']:
    TRANS[r].to_csv(f'colab37_transitions_{r}.csv')

summary = dict(collections={r: len(COLLECTION[r]) for r in COLLECTION},
               symbol_statistics=SYMSTATS.to_dict('records'),
               conditional_entropy=COND.to_dict('records'))
with open('colab37_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)
print(json.dumps(summary, indent=2))

print('\n--- Section 3.4 sentence, filled from the run above ---')
for _, x in SYMSTATS.iterrows():
    print(f"  {x.representation}: {x.entropy_bits} bits of {x.max_entropy_bits} "
          f"({x.normalised_entropy} normalised), top-5 share {x.top5_share}, "
          f"effective alphabet {x.effective_alphabet}")
print('\nDownload: colab37_summary.json + the two PNGs (appendix, note 32).')
